# Predicting joint contact forces from gait inputs: OpenSim JCF data, NIH ACL cohort

Runtime > Change runtime type > Python 3. CPU is enough for this notebook; a GPU comes later, for the model fitting.

## Setup

Clone the private GitHub repository `marysalvana/OpenSim-JCF-GP`. Cloning needs a GitHub personal access token with read access to the repository (GitHub > Settings > Developer settings > Fine-grained tokens). Paste it when asked; it is not stored.

In [ ]:
import os
from getpass import getpass

if not os.path.isdir("OpenSim-JCF-GP"):
    token = getpass("GitHub token (leave empty if the repository is public): ")
    url = f"https://{token}@github.com/marysalvana/OpenSim-JCF-GP.git" if token else "https://github.com/marysalvana/OpenSim-JCF-GP.git"
    os.system(f"git clone -q {url}")
os.listdir("OpenSim-JCF-GP")

In [ ]:
import sys
sys.path.insert(0, "OpenSim-JCF-GP/codes")
from jcfgp import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Data

Four CSV files in `data/`, defined column by column in `data/log.txt`. Read `log.txt` first.

One row is one stride of the right leg. `x-*.csv` holds the inputs (three ground reaction force curves, five joint angle curves, belt speed, mass, height), `y-*.csv` the outputs (the hip, knee and ankle joint contact force curves from OpenSim). Each curve is sampled at 101 points of the gait cycle, 0 to 100 %, heel strike to heel strike. Row i of x and row i of y are the same stride.

In [ ]:
print(open("OpenSim-JCF-GP/data/log.txt").read())

In [ ]:
x_train, y_train, x_test, y_test = load_data("OpenSim-JCF-GP/data")

## Explore

A data frame with one row per stride. The first three columns identify the stride, the next three are the scalar inputs, the rest are the curves, one column per gait point.

In [ ]:
x_train.shape, y_train.shape, x_test.shape, y_test.shape

In [ ]:
x_train.columns[:12].tolist()

In [ ]:
x_train.iloc[:5, :10]

In [ ]:
y_train.columns[:8].tolist()

In [ ]:
x_train["subject"].value_counts().sort_index()

In [ ]:
x_test["subject"].value_counts().sort_index()

In [ ]:
x_train["belt_speed_mps"].value_counts().sort_index()

In [ ]:
x_train.groupby("subject")[["mass_kg", "height_m"]].first()

## From wide columns to curves

A curve called `name` occupies the 101 columns `name_t000` to `name_t100`. `curve` returns it as a `(strides, 101)` array.

In [ ]:
grf_v = curve(x_train, "grf_vertical_BW")
grf_v.shape

In [ ]:
grf_v[0, [0, 15, 50, 62, 100]]  # one stride: heel strike, loading peak, push-off peak, about toe-off, next heel strike

In [ ]:
knee = curve(y_train, "knee_jcf_BW")
knee.max(axis=1)[:10]  # peak knee JCF of the first ten strides, body weights

## Visualise the inputs

All 480 training strides of each input curve, coloured by belt speed, with the mean curve of each speed on top. Stance is roughly the first 62 % of the cycle (dotted line), swing the rest. Look for: the two humps of the vertical GRF, braking then propulsion in the fore-aft GRF, hip extension until about 50 %, the knee flexion peak in swing, and the ankle push-off near 60 %.

In [ ]:
overlay_grid(x_train, INPUT_CURVES, title="Inputs: 480 training strides, 12 subjects, coloured by belt speed");

In [ ]:
overlay_grid(x_test, ["grf_vertical_BW", "knee_flexion_deg", "ankle_angle_deg"], ncols=3, title="The same inputs for the 3 test subjects");

## Visualise the outputs

The joint contact force is the total force pressing the two bones of a joint together: the external load plus the compression produced by the muscles that cross the joint. That is why it reaches 3 to 6 body weights in walking while the ground reaction force is about 1 body weight. Two peaks in stance (loading response and push-off), near zero in swing.

In [ ]:
overlay_grid(y_train, OUTPUT_CURVES, ncols=3, title="Outputs: joint contact forces of the same 480 training strides");

In [ ]:
peaks = pd.DataFrame({name.split("_")[0]: curve(y_train, name).max(axis=1) for name in OUTPUT_CURVES})
peaks.describe().round(2)  # peak JCF per stride, body weights

## One stride, inputs and outputs together

The question the model has to answer: given the eight curves and the three scalars, produce the three JCF curves. Same stride, same gait axis.

In [ ]:
show_stride(x_train, y_train, i=0);

## Subject to subject

The same joint, one panel per training subject, strides at 1.0 m/s only. Each subject has a recognisable shape of their own; that between-subject variation, not stride-to-stride noise, is what makes prediction for a new person hard.

In [ ]:
per_subject(x_train, y_train, "knee_jcf_BW", speed=1.0);